# CAE Flight Simulator Manufacturing — Solution Installer

This notebook deploys the complete demo into your Fabric workspace.  
It uses [`fabric-cicd`](https://microsoft.github.io/fabric-cicd/) to publish items directly from a GitHub repo.

## Prerequisites
| Requirement | Details |
|---|---|
| Fabric capacity | F2+ (F16+ for AI features) |
| Workspace permissions | Contributor or Admin |

## What gets deployed
| Category | Items |
|---|---|
| **Lakehouse** | `CAEManufacturing_LH` — staging area for CSVs |
| **Notebooks** | GetStarted, PostDeploymentConfig, LoadData, 3 Simulators, Agent |

After `fabric-cicd` publishes, the **PostDeploymentConfig** notebook creates the SQL Database tables and loads all data.

**Click Run All to install.**

In [ ]:
# === CONFIGURATION ===
REPO_URL  = "https://github.com/benoit-d/cae-demo.git"   # <-- your repo
REPO_REF  = "main"                                        # tag or branch
WORKSPACE_PATH = "workspace"                              # folder with Fabric items
DATA_PATH      = "data"                                   # folder with CSV seed data

ITEM_TYPES_IN_SCOPE = [
    "Notebook", "Lakehouse", "Environment",
    "Eventhouse", "Eventstream",
    "KQLDatabase", "KQLDashboard", "KQLQueryset",
    "Reflex", "SemanticModel", "Report",
    "SQLDatabase",
]

In [ ]:
# Step 1 — Install packages
%pip install -q fabric-cicd azure-identity gitpython

In [ ]:
# Step 2 — Clone the repo
import os, shutil, tempfile
from git import Repo

clone_dir = os.path.join(tempfile.gettempdir(), "cae-demo-install")
if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)

print(f"Cloning {REPO_URL} @ {REPO_REF} ...")
Repo.clone_from(REPO_URL, clone_dir, branch=REPO_REF, depth=1)

workspace_dir = os.path.join(clone_dir, WORKSPACE_PATH)
data_dir      = os.path.join(clone_dir, DATA_PATH)
assert os.path.isdir(workspace_dir)
assert os.path.isdir(data_dir)
print("Clone OK")

In [ ]:
# Step 3 — Publish Fabric items
from fabric_cicd import FabricWorkspace, publish_all_items

WORKSPACE_ID = ""  # leave blank to auto-detect inside a Fabric notebook

try:
    import notebookutils
    if not WORKSPACE_ID:
        WORKSPACE_ID = notebookutils.fabric.get_workspace_id()
except ImportError:
    if not WORKSPACE_ID:
        raise ValueError("Set WORKSPACE_ID when running outside Fabric.")

try:
    from notebookutils.credentials import getToken
    from azure.core.credentials import AccessToken
    class _FabricCred:
        def get_token(self, *s, **k):
            return AccessToken(getToken("https://api.fabric.microsoft.com"), 0)
    token_credential = _FabricCred()
except ImportError:
    from azure.identity import AzureCliCredential
    token_credential = AzureCliCredential()

ws = FabricWorkspace(
    workspace_id=WORKSPACE_ID,
    repository_directory=workspace_dir,
    item_type_in_scope=ITEM_TYPES_IN_SCOPE,
    token_credential=token_credential,
)
print(f"Publishing to workspace {WORKSPACE_ID} ...")
publish_all_items(ws)
print("All Fabric items published.")

In [ ]:
# Step 4 — Upload seed data to the staging Lakehouse
import glob

try:
    import notebookutils
    ws_items = notebookutils.fabric.list_items()
    lh = next((i for i in ws_items if i.get('displayName') == 'CAEManufacturing_LH'), None)

    if lh:
        lh_id = lh['id']
        for folder in ['erp', 'hr', 'telemetry', 'plm']:
            src = os.path.join(data_dir, folder)
            if not os.path.isdir(src):
                continue
            for f in sorted(glob.glob(os.path.join(src, '*'))):
                fname = os.path.basename(f)
                dest = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
                        f"{lh_id}/Files/data/{folder}/{fname}")
                notebookutils.fs.cp(f"file://{f}", dest)
                print(f"  {folder}/{fname}")
        print("\nAll seed data uploaded.")
    else:
        print("CAEManufacturing_LH not found yet. Re-run after it appears.")
except ImportError:
    print("Upload data/ folder to CAEManufacturing_LH/Files/data/ manually.")

In [ ]:
# Step 5 — Clean up and next steps
shutil.rmtree(clone_dir, ignore_errors=True)

print("\n" + "=" * 60)
print("  INSTALLATION COMPLETE")
print("=" * 60)
print("\nNext step:")
print("  1. Open  PostDeploymentConfig  notebook")
print("  2. Run All — it creates SQL tables and loads all data")
print("  3. Open  GetStarted  for the guided walkthrough")

# CAE Flight Simulator Manufacturing — Solution Installer

This notebook deploys the complete demo into your Fabric workspace.  
It uses [`fabric-cicd`](https://microsoft.github.io/fabric-cicd/) to publish items directly from a GitHub repo.

## Prerequisites
| Requirement | Details |
|---|---|
| Fabric capacity | F2+ (F16+ for AI features) |
| Workspace permissions | Contributor or Admin |

## What gets deployed
| Category | Items |
|---|---|
| **Lakehouse** | `CAEManufacturing_LH` — staging area for CSVs |
| **Notebooks** | GetStarted, PostDeploymentConfig, LoadData, 3 Simulators, Agent |

After `fabric-cicd` publishes, the **PostDeploymentConfig** notebook creates the SQL Database tables and loads all data.

**Click Run All to install.**

In [ ]:
# === CONFIGURATION ===
REPO_URL  = "https://github.com/benoit-d/cae-demo.git"   # <-- your repo
REPO_REF  = "main"                                        # tag or branch
WORKSPACE_PATH = "workspace"                              # folder with Fabric items
DATA_PATH      = "data"                                   # folder with CSV seed data

ITEM_TYPES_IN_SCOPE = [
    "Notebook", "Lakehouse", "Environment",
    "Eventhouse", "Eventstream",
    "KQLDatabase", "KQLDashboard", "KQLQueryset",
    "Reflex", "SemanticModel", "Report",
    "SQLDatabase",
]

In [ ]:
# Step 1 — Install packages
%pip install -q fabric-cicd azure-identity gitpython

In [ ]:
# Step 2 — Clone the repo
import os, shutil, tempfile
from git import Repo

clone_dir = os.path.join(tempfile.gettempdir(), "cae-demo-install")
if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)

print(f"Cloning {REPO_URL} @ {REPO_REF} ...")
Repo.clone_from(REPO_URL, clone_dir, branch=REPO_REF, depth=1)

workspace_dir = os.path.join(clone_dir, WORKSPACE_PATH)
data_dir      = os.path.join(clone_dir, DATA_PATH)
assert os.path.isdir(workspace_dir)
assert os.path.isdir(data_dir)
print("Clone OK")

In [ ]:
# Step 3 — Publish Fabric items
from fabric_cicd import FabricWorkspace, publish_all_items

WORKSPACE_ID = ""  # leave blank to auto-detect inside a Fabric notebook

try:
    import notebookutils
    if not WORKSPACE_ID:
        WORKSPACE_ID = notebookutils.fabric.get_workspace_id()
except ImportError:
    if not WORKSPACE_ID:
        raise ValueError("Set WORKSPACE_ID when running outside Fabric.")

try:
    from notebookutils.credentials import getToken
    from azure.core.credentials import AccessToken
    class _FabricCred:
        def get_token(self, *s, **k):
            return AccessToken(getToken("https://api.fabric.microsoft.com"), 0)
    token_credential = _FabricCred()
except ImportError:
    from azure.identity import AzureCliCredential
    token_credential = AzureCliCredential()

ws = FabricWorkspace(
    workspace_id=WORKSPACE_ID,
    repository_directory=workspace_dir,
    item_type_in_scope=ITEM_TYPES_IN_SCOPE,
    token_credential=token_credential,
)
print(f"Publishing to workspace {WORKSPACE_ID} ...")
publish_all_items(ws)
print("All Fabric items published.")

In [ ]:
# Step 4 — Upload seed data to the staging Lakehouse
import glob

try:
    import notebookutils
    ws_items = notebookutils.fabric.list_items()
    lh = next((i for i in ws_items if i.get('displayName') == 'CAEManufacturing_LH'), None)

    if lh:
        lh_id = lh['id']
        for folder in ['erp', 'hr', 'telemetry', 'plm']:
            src = os.path.join(data_dir, folder)
            if not os.path.isdir(src):
                continue
            for f in sorted(glob.glob(os.path.join(src, '*'))):
                fname = os.path.basename(f)
                dest = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
                        f"{lh_id}/Files/data/{folder}/{fname}")
                notebookutils.fs.cp(f"file://{f}", dest)
                print(f"  {folder}/{fname}")
        print("\nAll seed data uploaded.")
    else:
        print("CAEManufacturing_LH not found yet. Re-run after it appears.")
except ImportError:
    print("Upload data/ folder to CAEManufacturing_LH/Files/data/ manually.")

In [ ]:
# Step 5 — Clean up and next steps
shutil.rmtree(clone_dir, ignore_errors=True)

print("\n" + "=" * 60)
print("  INSTALLATION COMPLETE")
print("=" * 60)
print("\nNext step:")
print("  1. Open  PostDeploymentConfig  notebook")
print("  2. Run All — it creates SQL tables and loads all data")
print("  3. Open  GetStarted  for the guided walkthrough")

# CAE Flight Simulator Manufacturing — Solution Installer

This notebook deploys the complete demo into your Fabric workspace.  
It uses [`fabric-cicd`](https://microsoft.github.io/fabric-cicd/) — the same library behind Fabric Jumpstart — to publish items directly from a GitHub repo.

## Prerequisites
| Requirement | Details |
|---|---|
| Fabric capacity | F2+ (F16+ for AI features) |
| Workspace permissions | Contributor or Admin |

## What gets deployed
| Category | Items |
|---|---|
| **SQL Database** | `CAEManufacturing_SQLDB` — projects, tasks, employees, HR, ERP |
| **Lakehouse** | `CAEManufacturing_LH` — staging area for CSVs |
| **Eventhouse** | `CAEManufacturingEH` — real-time telemetry & clock-in events |
| **Eventstreams** | `SimulatorTelemetryStream`, `ClockInEventStream` |
| **Notebooks** | GetStarted, PostDeploymentConfig, LoadData, Simulators, Agent |

**Click Run All to install.**

In [ ]:
# === CONFIGURATION ===
REPO_URL  = "https://github.com/benoit-d/cae-demo.git"   # <-- your repo
REPO_REF  = "main"                                        # tag or branch
WORKSPACE_PATH = "workspace"                              # folder with Fabric items
DATA_PATH      = "data"                                   # folder with CSV / JSON seed data

ITEM_TYPES_IN_SCOPE = [
    "Notebook", "Lakehouse", "Environment",
    "Eventhouse", "Eventstream",
    "KQLDatabase", "KQLDashboard", "KQLQueryset",
    "Reflex", "SemanticModel", "Report",
    "SQLDatabase",
]

In [ ]:
# Step 1 — Install packages
%pip install -q fabric-cicd azure-identity gitpython

In [ ]:
# Step 2 — Clone the repo
import os, shutil, tempfile
from git import Repo

clone_dir = os.path.join(tempfile.gettempdir(), "cae-demo-install")
if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)

print(f"Cloning {REPO_URL} @ {REPO_REF} …")
Repo.clone_from(REPO_URL, clone_dir, branch=REPO_REF, depth=1)

workspace_dir = os.path.join(clone_dir, WORKSPACE_PATH)
data_dir      = os.path.join(clone_dir, DATA_PATH)
assert os.path.isdir(workspace_dir)
assert os.path.isdir(data_dir)
print("Clone OK")

In [ ]:
# Step 3 — Publish Fabric items with fabric-cicd
from fabric_cicd import FabricWorkspace, publish_all_items

WORKSPACE_ID = ""  # leave blank to auto-detect in a Fabric notebook

try:
    import notebookutils
    if not WORKSPACE_ID:
        WORKSPACE_ID = notebookutils.fabric.get_workspace_id()
except ImportError:
    if not WORKSPACE_ID:
        raise ValueError("Set WORKSPACE_ID when running outside Fabric.")

try:
    from notebookutils.credentials import getToken
    from azure.core.credentials import AccessToken
    class _Cred:
        def get_token(self, *s, **k):
            return AccessToken(getToken("https://api.fabric.microsoft.com"), 0)
    token_credential = _Cred()
except ImportError:
    from azure.identity import AzureCliCredential
    token_credential = AzureCliCredential()

ws = FabricWorkspace(
    workspace_id=WORKSPACE_ID,
    repository_directory=workspace_dir,
    item_type_in_scope=ITEM_TYPES_IN_SCOPE,
    token_credential=token_credential,
)
print(f"Publishing to workspace {WORKSPACE_ID} …")
publish_all_items(ws)
print("All Fabric items published.")

In [ ]:
# Step 4 — Upload seed data to the staging Lakehouse
import glob

try:
    import notebookutils
    ws_items = notebookutils.fabric.list_items()
    lh = next((i for i in ws_items if i.get('displayName') == 'CAEManufacturing_LH'), None)

    if lh:
        lh_id = lh['id']
        for folder in ['erp', 'hr', 'telemetry', 'plm']:
            src = os.path.join(data_dir, folder)
            if not os.path.isdir(src):
                continue
            for f in sorted(glob.glob(os.path.join(src, '*'))):
                fname = os.path.basename(f)
                dest = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
                        f"{lh_id}/Files/data/{folder}/{fname}")
                notebookutils.fs.cp(f"file://{f}", dest)
                print(f"  {folder}/{fname}")
        print("\nAll seed data uploaded to CAEManufacturing_LH.")
    else:
        print("CAEManufacturing_LH not found. Upload data manually after it appears.")
except ImportError:
    print("Upload data/ contents to CAEManufacturing_LH/Files/data/ manually.")

In [ ]:
# Step 5 — Clean up
shutil.rmtree(clone_dir, ignore_errors=True)

print("\n" + "=" * 60)
print("  INSTALLATION COMPLETE")
print("=" * 60)
print("\nNext step: Open PostDeploymentConfig → Run All")
print("           That notebook creates the SQL tables and loads all data.")

# CAE Flight Simulator Manufacturing - Solution Installer

This notebook deploys the complete **CAE Flight Simulator Manufacturing** demo into your Fabric workspace.

It is **fully self-contained** — no dependency on Fabric Jumpstart. It uses:
- [`fabric-cicd`](https://microsoft.github.io/fabric-cicd/) to deploy Fabric items (notebooks, lakehouses, eventhouses, etc.)
- `notebookutils` to upload data files to the Lakehouse
- A GitHub clone of the source repo for all assets

## Prerequisites
- Fabric workspace with **F16+ capacity** (for AI features)
- **Contributor or Admin** permissions on the workspace

## What gets deployed

| Category | Items |
|---|---|
| **Data Stores** | `ManufacturingERP_LH`, `HRData_LH` (Lakehouses) |
| **Real-Time** | `CAEManufacturingEH` (Eventhouse), `SimulatorTelemetryStream`, `ClockInEventStream` (Eventstreams) |
| **Notebooks** | GetStarted, LoadERPData, LoadHRData, SimulatorTelemetryEmulator, ClockInEventEmulator, CapacityManagementAgent, PostDeploymentConfig |
| **Analytics** | `ManufacturingFloorDashboard` (KQL Dashboard), `ManufacturingQueries` (KQL Queryset) |
| **Alerts** | `MaintenanceActivator` (Reflex) |
| **Runtime** | `CAEDemoRuntime` (Environment) |

**Click Run All to start the installation.**

In [ ]:
# ============================================================
# CONFIGURATION - Update these values for your environment
# ============================================================

# GitHub repository containing the demo assets
REPO_URL = "https://github.com/YOUR_ORG/cae-flight-simulator-manufacturing.git"
REPO_REF = "v1.0.0"  # Tag or branch to deploy from

# Workspace path inside the repo that contains Fabric items
WORKSPACE_PATH = "workspace"

# Data files path inside the repo
DATA_PATH = "data"

# Fabric item types to deploy
ITEM_TYPES_IN_SCOPE = [
    "Notebook",
    "Lakehouse",
    "Eventhouse",
    "Eventstream",
    "KQLDatabase",
    "KQLDashboard",
    "KQLQueryset",
    "Reflex",
    "Environment",
    "SemanticModel",
    "Report",
]

# Target Lakehouse for data file upload
DATA_DESTINATION_LAKEHOUSE = "ManufacturingERP_LH"
DATA_DESTINATION_PATH = "data"  # Files/ subfolder in the Lakehouse

In [ ]:
# ============================================================
# Step 1: Install required packages
# ============================================================
%pip install -q fabric-cicd azure-identity gitpython

In [ ]:
# ============================================================
# Step 2: Clone the source repository
# ============================================================
import os
import shutil
import tempfile
from git import Repo

# Clone into a temp directory
clone_dir = os.path.join(tempfile.gettempdir(), "cae-demo-install")

# Clean up any previous clone
if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)

print(f"Cloning {REPO_URL} (ref: {REPO_REF})...")
repo = Repo.clone_from(
    REPO_URL,
    clone_dir,
    branch=REPO_REF,
    depth=1  # Shallow clone for speed
)
print(f"Repository cloned to {clone_dir}")

# Verify key directories exist
workspace_dir = os.path.join(clone_dir, WORKSPACE_PATH)
data_dir = os.path.join(clone_dir, DATA_PATH)

assert os.path.isdir(workspace_dir), f"Workspace directory not found: {workspace_dir}"
assert os.path.isdir(data_dir), f"Data directory not found: {data_dir}"

print(f"Workspace items directory: {workspace_dir}")
print(f"Data files directory: {data_dir}")

# List items to deploy
items = []
for root, dirs, files in os.walk(workspace_dir):
    for d in dirs:
        if '.' in d and not d.startswith('.'):
            rel = os.path.relpath(os.path.join(root, d), workspace_dir)
            items.append(rel)

print(f"\nFabric items found: {len(items)}")
for item in sorted(items):
    print(f"   {item}")

In [ ]:
# ============================================================
# Step 3: Deploy Fabric items using fabric-cicd
# ============================================================
from fabric_cicd import FabricWorkspace, publish_all_items

# --- Workspace ID ---
# When running in a Fabric notebook, auto-detect the current workspace.
# Otherwise, set WORKSPACE_ID manually below.
WORKSPACE_ID = ""  # <-- paste your workspace GUID here if running outside Fabric

try:
    import notebookutils
    if not WORKSPACE_ID:
        WORKSPACE_ID = notebookutils.fabric.get_workspace_id()
    print(f"Deploying to workspace: {WORKSPACE_ID}")
except ImportError:
    if not WORKSPACE_ID:
        raise ValueError("Set WORKSPACE_ID in the cell above when running outside a Fabric notebook.")

# --- Authentication ---
try:
    # Fabric notebook runtime
    from notebookutils.credentials import getToken
    from azure.core.credentials import AccessToken

    class FabricNotebookCredential:
        """Wraps notebookutils.credentials for the TokenCredential interface."""
        def get_token(self, *scopes, **kwargs):
            token = getToken("https://api.fabric.microsoft.com")
            return AccessToken(token, 0)

    token_credential = FabricNotebookCredential()
    print("Using Fabric notebook credential")
except ImportError:
    from azure.identity import AzureCliCredential
    token_credential = AzureCliCredential()
    print("Using Azure CLI credential")

# --- Deploy ---
target_workspace = FabricWorkspace(
    workspace_id=WORKSPACE_ID,
    repository_directory=workspace_dir,
    item_type_in_scope=ITEM_TYPES_IN_SCOPE,
    token_credential=token_credential,
)

print(f"\nPublishing Fabric items...")
publish_all_items(target_workspace)
print("\nAll Fabric items deployed!")

In [ ]:
# ============================================================
# Step 4: Upload data files to the Lakehouse
# ============================================================
# fabric-cicd deploys item definitions (shells) but not file content.
# We upload the CSV data files to the Lakehouse Files area so the
# data-loader notebooks can read them.

import glob

def upload_folder_to_lakehouse(source_folder, lakehouse_id, dest_subfolder):
    """Upload all files in source_folder to a Lakehouse Files path."""
    for csv_file in sorted(glob.glob(os.path.join(source_folder, '*.csv'))):
        file_name = os.path.basename(csv_file)
        dest = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
                f"{lakehouse_id}/Files/{dest_subfolder}/{file_name}")
        notebookutils.fs.cp(f"file://{csv_file}", dest)
        print(f"   {dest_subfolder}/{file_name}")


try:
    import notebookutils

    # Discover deployed lakehouses by name
    ws_items = notebookutils.fabric.list_items()
    lh_erp = next((i for i in ws_items if i.get('displayName') == 'ManufacturingERP_LH'), None)
    lh_hr  = next((i for i in ws_items if i.get('displayName') == 'HRData_LH'), None)

    print("Uploading data files...\n")

    if lh_erp:
        upload_folder_to_lakehouse(os.path.join(data_dir, 'erp'), lh_erp['id'], 'data/erp')
        upload_folder_to_lakehouse(os.path.join(data_dir, 'telemetry'), lh_erp['id'], 'data/telemetry')
    else:
        print("   ManufacturingERP_LH not found — upload ERP/telemetry CSVs manually.")

    if lh_hr:
        upload_folder_to_lakehouse(os.path.join(data_dir, 'hr'), lh_hr['id'], 'data/hr')
    elif lh_erp:
        print("   HRData_LH not found — uploading HR files to ManufacturingERP_LH.")
        upload_folder_to_lakehouse(os.path.join(data_dir, 'hr'), lh_erp['id'], 'data/hr')
    else:
        print("   HRData_LH not found — upload HR CSVs manually.")

    print("\nAll data files uploaded!")

except ImportError:
    print("notebookutils not available (running outside Fabric).")
    print(f"Upload the data/ folder contents manually to each Lakehouse's Files area.")
    print(f"  ERP + telemetry data -> ManufacturingERP_LH/Files/data/")
    print(f"  HR data             -> HRData_LH/Files/data/")

In [ ]:
# ============================================================
# Step 5: Clean up
# ============================================================
if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)
    print(f"Cleaned up temp directory.")

print()
print("=" * 60)
print("  INSTALLATION COMPLETE")
print("=" * 60)
print()
print("Next steps:")
print("  1. Open PostDeploymentConfig notebook -> Run All")
print("  2. Open GetStarted notebook for the guided walkthrough")
print("  3. Start Simulation/SimulatorTelemetryEmulator")
print("  4. Start Simulation/ClockInEventEmulator")
print("  5. Open the ManufacturingFloorDashboard")
print("  6. Open Agent/CapacityManagementAgent for AI scenarios")